In [13]:
%pip install -q pyhmmer
import os
from pathlib import Path

# базовая директория проекта — от неё строятся все пути
base_dir = Path("/home/bulat_kharisov/inf")
for sub in ("data", "hmm", "results"):
    (base_dir / sub).mkdir(parents=True, exist_ok=True)
os.chdir(base_dir)  # относительные пути в !-командах тоже пойдут от base_dir

import pyhmmer
print("base_dir:", base_dir)
print("cwd:", os.getcwd())
print("pyhmmer", pyhmmer.__version__)


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
base_dir: /home/bulat_kharisov/inf
cwd: /home/bulat_kharisov/inf
pyhmmer 0.12.1


In [14]:
GENESET="https://ftp.ebi.ac.uk/pub/ensemblorganisms/Eledone_cirrhosa/GCA_964016885.1/ensembl/geneset/2024_11"

!wget -q -O data/pep.fa.gz   $GENESET/pep.fa.gz
!wget -q -O data/genes.gff3.gz $GENESET/genes.gff3.gz
!gunzip -kf data/pep.fa.gz data/genes.gff3.gz

print("белков в протеоме:")
!grep -c '^>' data/pep.fa
print("пример заголовка:")
!head -1 data/pep.fa

белков в протеоме:
21014
пример заголовка:
>ENSHJAP00000000441.1 pep xcEleCirr1.1:1:35891292:35981699:1 gene:ENSHJAG00000000421.1 transcript:ENSHJAT00000000602.1 gene_biotype:protein_coding transcript_biotype:protein_coding


In [15]:
GENOME="https://ftp.ebi.ac.uk/pub/ensemblorganisms/Eledone_cirrhosa/GCA_964016885.1/genome"
# геном ~ГБ; -c докачивает при обрыве. Для пунктов 1-2 (hmmer) не нужен, понадобится в пункте 3.
!wget -c -O data/genome.fa.gz $GENOME/unmasked.fa.gz
!gunzip -kf data/genome.fa.gz
!ls -lh data/

--2026-06-09 15:05:22--  https://ftp.ebi.ac.uk/pub/ensemblorganisms/Eledone_cirrhosa/GCA_964016885.1/genome/unmasked.fa.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 761411582 (726M) [application/x-gzip]
Saving to: ‘data/genome.fa.gz’

data/genome.fa.gz   100%[===================>] 726.14M  3.15MB/s    in 3m 57s  

2026-06-09 15:09:19 (3.06 MB/s) - ‘data/genome.fa.gz’ saved [761411582/761411582]

total 3.7G
-rw-rw-r-- 1 bulat_kharisov bulat_kharisov  70M Apr  6  2025 genes.gff3
-rw-rw-r-- 1 bulat_kharisov bulat_kharisov 5.9M Apr  6  2025 genes.gff3.gz
-rw-rw-r-- 1 bulat_kharisov bulat_kharisov 2.9G Jul 18  2025 genome.fa
-rw-rw-r-- 1 bulat_kharisov bulat_kharisov 727M Jul 18  2025 genome.fa.gz
-rw-rw-r-- 1 bulat_kharisov bulat_kharisov  16M Jul 18  2025 pep.fa
-rw-rw-r-- 1 bulat_kharisov bulat_kharisov 5.3M Jul 18  2025 pep.fa.gz


## Скачать 10 профилей Pfam и собрать в один HMM-файл

10 семейств из листа selected, покрывают метилирование ДНК, модификации гистонов и гистон-подобные белки. Профили тянем по одному через InterPro API (`?annotation=hmm`) — легче, чем качать весь `Pfam-A.hmm`. В каждом профиле уже зашит порог GA (gathering), его и используем при поиске.

In [16]:
# Pfam-аккешн -> человекочитаемое имя семейства (используется и при сборке таблицы)
PFAMS = {
    "PF00145": "DNA C5-метилтрансфераза (DNMT1/3)",
    "PF01429": "Methyl-CpG-binding (MBD/MeCP2)",
    "PF12851": "TET-диоксигеназа",
    "PF00856": "SET (гистон-метилтрансфераза)",
    "PF02373": "JmjC (деметилаза гистонов)",
    "PF00850": "HDAC (гистондеацетилаза)",
    "PF01853": "MYST (гистон-ацетилтрансфераза)",
    "PF00439": "Bromodomain (ридер ацетил-лизина)",
    "PF00385": "Chromodomain (ридер метил-лизина)",
    "PF00125": "Core histone (гистоновый фолд)",
}

for pf in PFAMS:
    !wget -q -O hmm/{pf}.hmm.gz "https://www.ebi.ac.uk/interpro/wwwapi/entry/pfam/{pf}?annotation=hmm"

!gunzip -f hmm/*.hmm.gz
!cat hmm/PF*.hmm > hmm/epigenetics.hmm
print("профилей в файле:")
!grep -c '^NAME' hmm/epigenetics.hmm

профилей в файле:
10


## Запуск hmmsearch по протеому (пункт 2)

Прогоняем все 10 профилей по `pep.fa` через pyhmmer с порогами Pfam (`bit_cutoffs="gathering"` = `--cut_ga`). Для каждого профиля собираем прошедшие порог хиты белков. Сырые хиты сохраняем в `results/hits_raw.csv`.

In [17]:
import os, pyhmmer, pandas as pd
from pyhmmer.easel import SequenceFile
from pyhmmer.plan7 import HMMFile

# протеом и профили
with SequenceFile("data/pep.fa", digital=True) as sf:
    seqs = sf.read_block()
with HMMFile("hmm/epigenetics.hmm") as hf:
    hmms = list(hf)
print("белков:", len(seqs), "| профилей:", len(hmms))

def query_acc(top):
    q = getattr(top, "query", None)
    raw = (q.accession or q.name) if q is not None else (top.query_accession or top.query_name)
    return (raw.decode() if isinstance(raw, (bytes, bytearray)) else raw)

rows = []
for top in pyhmmer.hmmsearch(hmms, seqs, bit_cutoffs="gathering", cpus=os.cpu_count() or 2):
    pf = query_acc(top).split(".")[0]
    for hit in top:
        if hit.included:  # прошёл порог GA
            rows.append({"pfam": pf, "protein": hit.name,
                         "evalue": hit.evalue, "score": hit.score})

hits = pd.DataFrame(rows)
hits.to_csv("results/hits_raw.csv", index=False)
print("хитов прошло порог:", len(hits))

белков: 21014 | профилей: 10
хитов прошло порог: 194


### Таблица «семейство — ген — координаты» (итог пункта 2)

- из заголовков `pep.fa`: `protein_id -> gene_id` (поле `gene:`) и символ гена;
- из `genes.gff3`: `gene_id -> (chrom, start, end, strand)` по строкам с типом `gene`;
- хиты — из `hits` (результат pyhmmer, `results/hits_raw.csv`).

In [18]:
import pandas as pd, re

# 1) protein_id -> ген (из заголовков pep.fa)
pep2gene = {}
with open("data/pep.fa") as fh:
    for line in fh:
        if not line.startswith(">"):
            continue
        h = line[1:].strip()
        pid = h.split()[0]
        gene = re.search(r"gene:(\S+)", h)
        sym = re.search(r"gene_symbol:(\S+)", h)
        pep2gene[pid] = {"gene": gene.group(1) if gene else None,
                         "symbol": sym.group(1) if sym else ""}

# 2) gene_id -> координаты (из GFF3, строки type == gene)
gene_coords = {}
with open("data/genes.gff3") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        f = line.rstrip("\n").split("\t")
        if len(f) < 9 or f[2] != "gene":
            continue
        gid = re.search(r"ID=(?:gene:)?([^;]+)", f[8])
        name = re.search(r"Name=([^;]+)", f[8])
        if not gid:
            continue
        gene_coords[gid.group(1)] = {
            "chrom": f[0], "start": int(f[3]), "end": int(f[4]),
            "strand": f[6], "name": name.group(1) if name else "",
        }

# 3) хиты pyhmmer (из памяти или из results/hits_raw.csv)
hits = pd.read_csv("results/hits_raw.csv")

# обогащаем геном и координатами
# в pep.fa ген с версией (ENSHJAG...x.1), в GFF — без версии; матчим по обоим вариантам
def gc(gid):
    return gene_coords.get(gid) or gene_coords.get(str(gid).split(".")[0]) or {}

def coord_str(gid):
    co = gc(gid)
    return f"{co['chrom']}:{co['start']}-{co['end']}({co['strand']})" if co else ""

hits["family"]   = hits["pfam"].map(PFAMS).fillna(hits["pfam"])
hits["gene"]     = hits["protein"].map(lambda p: (pep2gene.get(p) or {}).get("gene") or "")
hits["gene_name"] = hits.apply(
    lambda r: (pep2gene.get(r.protein) or {}).get("symbol")
              or gc(r.gene).get("name", ""), axis=1)
hits["coords"]   = hits["gene"].map(coord_str)

# одна строка на (семейство, ген) — лучший по score
table = (hits.sort_values("score", ascending=False)
              .drop_duplicates(["family", "gene"])
              [["family", "pfam", "gene", "gene_name", "coords", "evalue", "score"]]
              .sort_values(["family", "score"], ascending=[True, False])
              .reset_index(drop=True))

table.to_csv("results/epigenetics_families.csv", index=False)

found = set(table.loc[table["gene"] != "", "pfam"])
missing = [f"{pf} ({name})" for pf, name in PFAMS.items() if pf not in found]
print(f"найдено семейств: {len(found)}/{len(PFAMS)}")
if missing:
    print("без находок:", "; ".join(missing))
table

найдено семейств: 10/10


,family,pfam,gene,gene_name,coords,evalue,score
0,Bromodomain (ридер ацетил-лизина),PF00439,ENSHJAG00000001122.1,,24:2837350-3147803(+),9.764343e-110,358.971710
1,Bromodomain (ридер ацетил-лизина),PF00439,ENSHJAG00000008793.1,PHIP,17:42116338-42155488(-),4.844654e-44,148.436218
2,Bromodomain (ридер ацетил-лизина),PF00439,ENSHJAG00000006509.1,,10:23919866-23996114(+),5.817987e-44,148.181412
3,Bromodomain (ридер ацетил-лизина),PF00439,ENSHJAG00000009888.1,,4:64749356-64816507(-),2.956474e-39,133.100037
4,Bromodomain (ридер ацетил-лизина),PF00439,ENSHJAG00000008006.1,,15:21393870-21430185(+),1.421288e-26,92.458183
...,...,...,...,...,...,...,...
122,SET (гистон-метилтрансфераза),PF00856,ENSHJAG00000012212.1,SP3,3:70818128-70866497(+),2.824427e-05,24.713785
123,SET (гистон-метилтрансфераза),PF00856,ENSHJAG00000014582.1,,3:90647957-90793104(+),3.041895e-05,24.610245
124,SET (гистон-метилтрансфераза),PF00856,ENSHJAG00000013431.1,RNF180,5:50398641-50402434(+),1.264806e-04,22.621073
125,SET (гистон-метилтрансфераза),PF00856,ENSHJAG00000013488.1,nfat5b,20:18570716-18581633(-),1.291724e-04,22.591677


In [19]:
pd.read_csv("/home/bulat_kharisov/inf/results/epigenetics_families.csv")

,family,pfam,gene,gene_name,coords,evalue,score
0,Bromodomain (ридер ацетил-лизина),PF00439,ENSHJAG00000001122.1,NaN,24:2837350-3147803(+),9.764343e-110,358.971710
1,Bromodomain (ридер ацетил-лизина),PF00439,ENSHJAG00000008793.1,PHIP,17:42116338-42155488(-),4.844654e-44,148.436218
2,Bromodomain (ридер ацетил-лизина),PF00439,ENSHJAG00000006509.1,NaN,10:23919866-23996114(+),5.817987e-44,148.181412
3,Bromodomain (ридер ацетил-лизина),PF00439,ENSHJAG00000009888.1,NaN,4:64749356-64816507(-),2.956474e-39,133.100037
4,Bromodomain (ридер ацетил-лизина),PF00439,ENSHJAG00000008006.1,NaN,15:21393870-21430185(+),1.421288e-26,92.458183
...,...,...,...,...,...,...,...
122,SET (гистон-метилтрансфераза),PF00856,ENSHJAG00000012212.1,SP3,3:70818128-70866497(+),2.824427e-05,24.713785
123,SET (гистон-метилтрансфераза),PF00856,ENSHJAG00000014582.1,NaN,3:90647957-90793104(+),3.041895e-05,24.610245
124,SET (гистон-метилтрансфераза),PF00856,ENSHJAG00000013431.1,RNF180,5:50398641-50402434(+),1.264806e-04,22.621073
125,SET (гистон-метилтрансфераза),PF00856,ENSHJAG00000013488.1,nfat5b,20:18570716-18581633(-),1.291724e-04,22.591677
